In [3]:
import json, random, os

# prende la prima colonna di ogni riga da train.txt
ids = [l.split()[0] for l in open("train.txt") if l.strip()]

# Recupera i 200 ID già estratti/scrapati per escluderli
already_scraped = set()
for fname in ["input_pilot_200.json", "output.json", "input.json"]:
    if os.path.exists(fname):
        try:
            data = json.load(open(fname, encoding="utf-8"))
            if isinstance(data, dict) and "tweetIds" in data:
                already_scraped.update(str(x) for x in data["tweetIds"])
            elif isinstance(data, list):
                already_scraped.update(str(it.get("tweet_id") or it.get("id_str")) for it in data if isinstance(it, dict))
        except Exception:
            pass

# Esclude gli ID già estratti
remaining_ids = [i for i in ids if i not in already_scraped]

random.seed(42)                                # riproducibile
sample_4800 = random.sample(remaining_ids, min(4800, len(remaining_ids)))

# Salva i nuovi 4800 ID in input.json per il prossimo scrape
json.dump({"tweetIds": sample_4800}, open("input.json", "w"), indent=2)

print(f"ID totali in train.txt: {len(ids)}")
print(f"ID già estratti ed esclusi: {len(already_scraped)}")
print(f"Nuovi ID salvati in input.json: {len(sample_4800)}")


ID totali in train.txt: 13669
ID già estratti ed esclusi: 200
Nuovi ID salvati in input.json: 4800


In [7]:
import json, re, html, os
import pandas as pd

# Supporta uno o più file di scraping se presenti
input_files = [f for f in ["output.json", "output2_200-4800.json"] if os.path.exists(f)]
if not input_files:
    input_files = ["output.json"]
IDS, N_ANNOT = "input.json", 100   # N_ANNOT = tweet per regione

raw = []
for f in input_files:
    data = json.load(open(f, encoding="utf-8"))
    if isinstance(data, dict):
        data = next(v for v in data.values() if isinstance(v, list))
    raw.extend(data)

def clean(t):
    t = html.unescape(t)
    t = re.sub(r"https?://\S+", "<URL>", t)
    t = re.sub(r"@\w+", "@USER", t)
    return re.sub(r"\s+", " ", t).strip()

rows = []
for it in raw:
    if it.get("error") or not it.get("text"):
        continue
    start = (it.get("display_text_range") or [0])[0]
    full = (it.get("place") or {}).get("full_name") or ""
    city, _, reg = full.partition(", ")
    p = it.get("parent") or it.get("quoted_tweet") or {}
    pstart = (p.get("display_text_range") or [0])[0]
    rows.append({
        "id": str(it.get("tweet_id") or it.get("id_str")),
        "text": clean(it["text"][start:]),
        "region": reg or None,
        "city": city or None,
        "is_reply": it.get("in_reply_to_status_id_str") is not None,
        "parent_id": it.get("in_reply_to_status_id_str") or (p.get("id_str") if p else None),
        "has_parent_text": bool(p.get("text")),
        "parent_text": clean(p["text"][pstart:]) if p.get("text") else None,
    })

df = pd.DataFrame(rows).drop_duplicates("id")
df.to_csv("clean.csv", index=False, encoding="utf-8")

# ---- statistiche per la slide ----
n_req = len(json.load(open(IDS))["tweetIds"]) if os.path.exists(IDS) else None
print(f"Recuperati: {len(df)}" + (f" / {n_req} ({len(df)/n_req:.0%})" if n_req else ""))
print("\nPer regione:\n", df["region"].value_counts(dropna=False).head(10))
target = df[df["region"].isin(["Lazio", "Campania"])]
print(f"\nLazio+Campania: {len(target)} ({len(target)/len(df):.0%} dei recuperati)")
print(f"Risposte (in_reply_to): {df['is_reply'].mean():.0%}")

# dopo aver creato df e target
print(f"Con ID del padre o citato: {df['parent_id'].notna().mean():.0%}")
print(f"Con testo del padre o citato: {df['has_parent_text'].mean():.0%}")
print(f"Lazio+Campania con testo di contesto: {target['has_parent_text'].sum()} / {len(target)}")

# ---- file di annotazione (stessi tweet per entrambi, per il kappa) ----
ctx = target[target.has_parent_text]
if not ctx.empty:
    sample = (pd.concat([g.sample(min(N_ANNOT, len(g)), random_state=42) for _, g in ctx.groupby("region")])
                .sample(frac=1, random_state=1))       # mescola le regioni
else:
    sample = ctx.copy()

for col in ["dialetto", "sarcasmo", "contesto_necessario"]:
    sample[col] = ""

for i, who in enumerate(["A", "B"]):
    # Seed differenti per annotatore e per pass per evitare bias d'ordine
    seed_p1 = 42 + i
    seed_p2 = 100 + i

    p1 = sample[["id", "region", "text"]].sample(frac=1, random_state=seed_p1).copy()
    for c in ["dialetto", "sarcasmo_senza_contesto"]:
        p1[c] = ""
    p1.to_csv(f"pass1_{who}.csv", index=False)

    p2 = sample[["id", "text", "parent_text"]].sample(frac=1, random_state=seed_p2).copy()
    p2["sarcasmo_con_contesto"] = ""
    p2.to_csv(f"pass2_{who}.csv", index=False)

print(f"\nCreati pass1_A/B.csv e pass2_A/B.csv ({len(sample)} tweet con ordini indipendenti)")

Recuperati: 4262 / 4800 (89%)

Per regione:
 region
Lazio             1743
Campania           993
Veneto             215
Calabria           115
Lombardy           113
Emilia Romagna     113
Lombardia          113
Sicily              85
Liguria             79
Sardegna            78
Name: count, dtype: int64

Lazio+Campania: 2736 (64% dei recuperati)
Risposte (in_reply_to): 4%
Con ID del padre o citato: 10%
Con testo del padre o citato: 9%
Lazio+Campania con testo di contesto: 234 / 2736

Creati pass1_A/B.csv e pass2_A/B.csv (178 tweet)
